# Laboratorio II – Generador de nombres de dinosaurios (RNN a nivel de caracteres)

**Ejecutar en Google Colab** (Entorno de ejecución → Cambiar tipo → GPU T4; también funciona en CPU, solo más lento).

Flujo del notebook:
1. Cargar `dinos.csv` → 2. Preprocesar → 3. Entrenar **dos configuraciones** → 4. Curvas de pérdida
→ 5. Muestreo (temperatura, top-k, top-p) → 6. Diez nombres nuevos → 7. Exportar el modelo para la app.

Todo lo que se guarda en `evidencias/` va al repositorio (sección 7 de la guía).

In [ ]:
import os, json, random, unicodedata, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("evidencias", exist_ok=True)
print("Dispositivo:", DEVICE, "| torch", torch.__version__)

## 0. Módulo del modelo
Esta celda crea `char_model.py`, **el mismo archivo** que usa la app web en `04_app/`.
Así el modelo que se entrena aquí y el que se carga en el servidor son idénticos.

In [ ]:
%%writefile char_model.py
"""
char_model.py
Modelo de lenguaje a nivel de caracteres (RNN / LSTM / GRU) + muestreo.
Lo usan el notebook de entrenamiento (01_generador_nombres) y la app web (04_app).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD, SOS, EOS = "<PAD>", "<SOS>", "<EOS>"


class CharRNN(nn.Module):
    """Embedding -> celda recurrente (rnn | lstm | gru) -> capa lineal sobre el vocabulario."""

    def __init__(self, vocab_size, emb_dim=32, hidden_size=128, num_layers=1,
                 cell="gru", dropout=0.0, pad_idx=0):
        super().__init__()
        celdas = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.rnn = celdas[cell.lower()](
            emb_dim, hidden_size, num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        # x: (batch, T) indices -> logits: (batch, T, vocab)
        salida, h = self.rnn(self.emb(x), h)
        return self.fc(salida), h


def filtrar_logits(logits, temperatura=1.0, top_k=0, top_p=1.0):
    """Aplica temperatura, top-k y top-p a un vector de logits (1D)."""
    logits = logits / max(float(temperatura), 1e-6)

    if top_k and top_k > 0:
        k = min(int(top_k), logits.size(-1))
        umbral = torch.topk(logits, k).values[-1]
        logits = logits.masked_fill(logits < umbral, float("-inf"))

    if top_p is not None and top_p < 1.0:
        ordenados, idx = torch.sort(logits, descending=True)
        probs = F.softmax(ordenados, dim=-1)
        acumulada_previa = probs.cumsum(-1) - probs
        # se quitan los caracteres que aparecen DESPUÉS de alcanzar p
        ordenados = ordenados.masked_fill(acumulada_previa >= top_p, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(0, idx, ordenados)

    return logits


@torch.no_grad()
def generar_nombre(modelo, stoi, itos, max_len=30, temperatura=1.0, top_k=0,
                   top_p=1.0, device="cpu"):
    """Generación autoregresiva: cada carácter muestreado es la entrada del siguiente paso."""
    modelo.eval()
    x = torch.tensor([[stoi[SOS]]], device=device)
    h, chars = None, []
    for _ in range(max_len):
        logits, h = modelo(x, h)
        logits = logits[0, -1].clone()
        logits[stoi[PAD]] = float("-inf")   # nunca generar PAD ni SOS
        logits[stoi[SOS]] = float("-inf")
        logits = filtrar_logits(logits, temperatura, top_k, top_p)
        idx = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
        if idx == stoi[EOS]:
            break
        chars.append(itos[idx])
        x = torch.tensor([[idx]], device=device)
    return "".join(chars)


def cargar_checkpoint(ruta, device="cpu"):
    """Carga el archivo dino_rnn.pt exportado por el notebook."""
    ck = torch.load(ruta, map_location=device, weights_only=False)
    cfg = ck["config"]
    modelo = CharRNN(
        vocab_size=len(ck["itos"]), emb_dim=cfg["emb"], hidden_size=cfg["oculto"],
        num_layers=cfg["capas"], cell=cfg["celda"], dropout=cfg.get("dropout", 0.0),
    ).to(device)
    modelo.load_state_dict(ck["state_dict"])
    modelo.eval()
    return modelo, ck

In [ ]:
from char_model import CharRNN, generar_nombre, PAD, SOS, EOS

## 1. Datos
Sube `dinos.csv` al panel de archivos de Colab (icono de carpeta) o móntalo desde Drive.
Revisa `df.head()`: si la columna de nombres tiene otro nombre, ajusta `COLUMNA`.

In [ ]:
RUTA_CSV = "dinos.csv"
df = pd.read_csv(RUTA_CSV)
print(df.shape); display(df.head())

candidatas = [c for c in df.columns if str(c).strip().lower() in ("name", "names", "nombre", "nombres", "dino", "dinosaur")]
COLUMNA = candidatas[0] if candidatas else df.columns[0]
print("Columna usada:", COLUMNA)

## 2. Preprocesamiento
Orden pedido por la guía: normalizar → vocabulario → tokens especiales → longitud máxima T → `<PAD>` → pares (X, Y) desplazados.

In [ ]:
def normalizar(texto):
    """Minúsculas, sin tildes, solo letras a-z (criterio único de escritura)."""
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z]", "", texto.lower().strip())

nombres = [normalizar(n) for n in df[COLUMNA].dropna()]
nombres = sorted(set(n for n in nombres if len(n) >= 2))
print(f"{len(nombres)} nombres únicos. Ejemplos:", nombres[:8])

# Vocabulario: especiales primero, <PAD> = 0
caracteres = sorted(set("".join(nombres)))
itos = [PAD, SOS, EOS] + caracteres
stoi = {c: i for i, c in enumerate(itos)}
PAD_IDX = stoi[PAD]
print("Tamaño del vocabulario:", len(itos))
print(stoi)

In [ ]:
# Tokens de inicio y fin: [<SOS>, x1, ..., xn, <EOS>]
secuencias = [[stoi[SOS]] + [stoi[c] for c in n] + [stoi[EOS]] for n in nombres]

# Longitud máxima: cada secuencia tiene T+1 tokens -> X e Y tienen T tokens
T = max(len(s) for s in secuencias) - 1
print("T =", T)

# Relleno con <PAD> hasta T+1
relleno = [s + [PAD_IDX] * (T + 1 - len(s)) for s in secuencias]
datos = torch.tensor(relleno, dtype=torch.long)

# Desplazamiento de un carácter: X = [x0..x_{T-1}], Y = [x1..x_T]
X, Y = datos[:, :-1], datos[:, 1:]
print("X:", X.shape, " Y:", Y.shape)
print("Ejemplo  ->", nombres[0])
print("X[0] =", [itos[i] for i in X[0].tolist()])
print("Y[0] =", [itos[i] for i in Y[0].tolist()])

In [ ]:
# División entrenamiento / validación (90 / 10)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(SEMILLA))
corte = int(0.9 * len(X))
idx_tr, idx_va = perm[:corte], perm[corte:]
ds_tr = TensorDataset(X[idx_tr], Y[idx_tr])
ds_va = TensorDataset(X[idx_va], Y[idx_va])
print(f"Entrenamiento: {len(ds_tr)} | Validación: {len(ds_va)}")

## 3. Modelo y entrenamiento — dos configuraciones
La pérdida es `CrossEntropyLoss(ignore_index=PAD)`: las posiciones `<PAD>` **no** cuentan en la pérdida.
Se guarda el estado con menor pérdida de validación (early stopping simple).

In [ ]:
CONFIGS = [
    {"nombre": "A_GRU_1x128",  "celda": "gru",  "capas": 1, "oculto": 128, "emb": 32,
     "dropout": 0.0, "epocas": 40, "lr": 3e-3, "batch": 64,
     "perdida": "CrossEntropyLoss(ignore_index=<PAD>)", "optimizador": "Adam"},
    {"nombre": "B_LSTM_2x256", "celda": "lstm", "capas": 2, "oculto": 256, "emb": 64,
     "dropout": 0.3, "epocas": 40, "lr": 1e-3, "batch": 64,
     "perdida": "CrossEntropyLoss(ignore_index=<PAD>)", "optimizador": "Adam"},
]

def correr_epoca(modelo, loader, criterio, optimizador=None):
    entrenando = optimizador is not None
    modelo.train(entrenando)
    total, tokens = 0.0, 0
    with torch.set_grad_enabled(entrenando):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, _ = modelo(xb)
            perdida = criterio(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            if entrenando:
                optimizador.zero_grad()
                perdida.backward()
                nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                optimizador.step()
            n = (yb != PAD_IDX).sum().item()      # tokens reales (sin PAD)
            total += perdida.item() * n
            tokens += n
    return total / tokens

def entrenar(cfg):
    torch.manual_seed(SEMILLA)
    modelo = CharRNN(len(itos), cfg["emb"], cfg["oculto"], cfg["capas"], cfg["celda"],
                     cfg["dropout"], PAD_IDX).to(DEVICE)
    criterio = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=cfg["lr"])
    dl_tr = DataLoader(ds_tr, batch_size=cfg["batch"], shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=256)
    hist = {"train": [], "val": []}
    mejor, mejor_estado, mejor_epoca = float("inf"), None, 0
    for ep in range(1, cfg["epocas"] + 1):
        tr = correr_epoca(modelo, dl_tr, criterio, optimizador)
        va = correr_epoca(modelo, dl_va, criterio)
        hist["train"].append(tr); hist["val"].append(va)
        if va < mejor:
            mejor, mejor_epoca = va, ep
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            print(f"[{cfg['nombre']}] época {ep:3d}  train={tr:.4f}  val={va:.4f}")
    modelo.load_state_dict(mejor_estado)
    params = sum(p.numel() for p in modelo.parameters())
    resumen = {**cfg, "parametros": params, "mejor_val": round(mejor, 4),
               "mejor_epoca": mejor_epoca, "train_final": round(hist["train"][-1], 4)}
    return modelo, hist, resumen

resultados = {}
for cfg in CONFIGS:
    modelo, hist, resumen = entrenar(cfg)
    resultados[cfg["nombre"]] = {"modelo": modelo, "hist": hist, "resumen": resumen}
    print(modelo, "\n")

## 4. Curvas de pérdida (entrenamiento y validación)

In [ ]:
fig, ejes = plt.subplots(1, len(resultados), figsize=(6 * len(resultados), 4), sharey=True)
for ax, (nombre, r) in zip(np.atleast_1d(ejes), resultados.items()):
    ax.plot(r["hist"]["train"], label="entrenamiento")
    ax.plot(r["hist"]["val"], label="validación")
    ax.axvline(r["resumen"]["mejor_epoca"] - 1, ls="--", c="gray", lw=1, label="mejor época")
    ax.set_title(nombre); ax.set_xlabel("época"); ax.set_ylabel("pérdida (entropía cruzada)")
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("evidencias/curvas_perdida.png", dpi=150); plt.show()

tabla = pd.DataFrame([r["resumen"] for r in resultados.values()])
tabla = tabla[["nombre", "celda", "capas", "oculto", "emb", "dropout", "epocas", "lr", "batch",
               "perdida", "optimizador", "parametros", "train_final", "mejor_val", "mejor_epoca"]]
display(tabla)
tabla.to_csv("evidencias/configuraciones.csv", index=False)
with open("evidencias/historial_perdidas.json", "w") as f:
    json.dump({k: v["hist"] for k, v in resultados.items()}, f)

In [ ]:
# Elegimos el modelo con menor pérdida de validación
NOMBRE_MEJOR = min(resultados, key=lambda k: resultados[k]["resumen"]["mejor_val"])
modelo = resultados[NOMBRE_MEJOR]["modelo"].eval()
print("Modelo elegido:", NOMBRE_MEJOR)

## 5. Muestreo: temperatura, top-k y top-p
Para cada estrategia se generan 100 nombres y se mide: % nuevos (no están en el dataset), % únicos y longitud media.

In [ ]:
ESTRATEGIAS = [
    {"estrategia": "T=0.5",          "temperatura": 0.5, "top_k": 0,  "top_p": 1.0},
    {"estrategia": "T=1.0",          "temperatura": 1.0, "top_k": 0,  "top_p": 1.0},
    {"estrategia": "T=1.5",          "temperatura": 1.5, "top_k": 0,  "top_p": 1.0},
    {"estrategia": "top-k=3",        "temperatura": 1.0, "top_k": 3,  "top_p": 1.0},
    {"estrategia": "top-k=10",       "temperatura": 1.0, "top_k": 10, "top_p": 1.0},
    {"estrategia": "top-p=0.5",      "temperatura": 1.0, "top_k": 0,  "top_p": 0.5},
    {"estrategia": "top-p=0.9",      "temperatura": 1.0, "top_k": 0,  "top_p": 0.9},
    {"estrategia": "T=0.8 + top-p=0.9", "temperatura": 0.8, "top_k": 0, "top_p": 0.9},
]
CONJUNTO = set(nombres)   # todo el dataset: un nombre "nuevo" no puede estar en él
N = 100

filas, muestras = [], {}
for e in ESTRATEGIAS:
    torch.manual_seed(SEMILLA)
    gen = [generar_nombre(modelo, stoi, itos, T + 5, e["temperatura"], e["top_k"], e["top_p"], DEVICE) for _ in range(N)]
    muestras[e["estrategia"]] = gen
    filas.append({**e,
                  "% nuevos": round(100 * np.mean([g not in CONJUNTO for g in gen]), 1),
                  "% únicos": round(100 * len(set(gen)) / N, 1),
                  "long. media": round(np.mean([len(g) for g in gen]), 1),
                  "ejemplos": ", ".join(gen[:5])})
tabla_muestreo = pd.DataFrame(filas)
display(tabla_muestreo)
tabla_muestreo.to_csv("evidencias/muestreo.csv", index=False)

**Cómo leer la tabla:** con temperatura baja (0.5) los nombres son muy parecidos a los reales y se repiten (menos únicos, menos nuevos);
con temperatura alta (1.5) aparecen combinaciones raras o impronunciables. Top-k y top-p recortan la cola de caracteres improbables:
top-k=3 y top-p=0.5 se comportan como una temperatura baja; top-p=0.9 conserva diversidad sin las combinaciones absurdas.
**Escribe aquí tu propio análisis con los números que obtuviste.**

## 6. Diez nombres nuevos y elección del dinosaurio

In [ ]:
MUESTREO_ELEGIDO = {"temperatura": 0.8, "top_k": 0, "top_p": 0.9}   # ajústalo según tu tabla

def nombres_nuevos(n=10, min_len=5, max_len=18, **muestreo):
    vistos, salida = set(), []
    while len(salida) < n:
        g = generar_nombre(modelo, stoi, itos, T + 5, device=DEVICE, **muestreo)
        if g not in CONJUNTO and g not in vistos and min_len <= len(g) <= max_len:
            vistos.add(g); salida.append(g.capitalize())
    return salida

torch.manual_seed(7)
diez = nombres_nuevos(10, **MUESTREO_ELEGIDO)
assert all(n.lower() not in CONJUNTO for n in diez)
for i, n in enumerate(diez, 1):
    print(f"{i:2d}. {n}")

In [ ]:
NOMBRE_ELEGIDO = diez[0]          # <-- cambia el índice por el que más te guste
print("Dinosaurio elegido:", NOMBRE_ELEGIDO)
with open("evidencias/nombres_generados.json", "w") as f:
    json.dump({"modelo": NOMBRE_MEJOR, "muestreo": MUESTREO_ELEGIDO,
               "nombres": diez, "elegido": NOMBRE_ELEGIDO}, f, indent=2, ensure_ascii=False)

## 7. Exportar el modelo para la app
Descarga `dino_rnn.pt` y cópialo en `04_app/model/`. Contiene pesos, vocabulario, configuración,
métricas, muestreo elegido y el conjunto de nombres reales (para que la app descarte copias).

In [ ]:
resumen = resultados[NOMBRE_MEJOR]["resumen"]
torch.save({
    "state_dict": {k: v.cpu() for k, v in modelo.state_dict().items()},
    "config": {k: resumen[k] for k in ("nombre", "celda", "capas", "oculto", "emb", "dropout",
                                       "epocas", "lr", "batch", "perdida", "optimizador")},
    "metricas": {"mejor_val": resumen["mejor_val"], "mejor_epoca": resumen["mejor_epoca"],
                 "train_final": resumen["train_final"], "parametros": resumen["parametros"]},
    "muestreo": MUESTREO_ELEGIDO,
    "stoi": stoi, "itos": itos, "T": T,
    "nombres_reales": sorted(CONJUNTO),
}, "dino_rnn.pt")
print("Guardado dino_rnn.pt  (", round(os.path.getsize("dino_rnn.pt") / 1e6, 2), "MB )")

In [ ]:
# Descarga rápida desde Colab
try:
    from google.colab import files
    !zip -qr evidencias.zip evidencias
    files.download("dino_rnn.pt"); files.download("evidencias.zip")
except ImportError:
    pass